In [1]:
import os
from dotenv import load_dotenv

import datasets
from diploma import evaluate
from langchain_community.embeddings import OctoAIEmbeddings
from ragas.metrics import faithfulness, context_recall, context_precision, answer_relevancy
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings, SentenceTransformerEmbeddings

In [2]:
dotenv_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path)

token = os.environ["OCTOAI_TOKEN"]
endpoint = os.environ["ENDPOINT"]

REPO_ID = "llmware/rag_instruct_benchmark_tester"
LIMIT = 2

dataset = datasets.load_dataset(path=REPO_ID, name=None, split="train")
dataset = dataset[:LIMIT]

In [ ]:
models = ["mistral-7b-instruct-fp16"]
judges = ["llama-2-70b-chat-fp16"]
chunk_sizes = [200,  500, 1500]
chunk_num = [1]
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embedding_function = OctoAIEmbeddings(endpoint_url=(endpoint + "/v1/embeddings"), octoai_api_token=token)
REPEAT = 1

result = evaluate.run_benchmark(
    dataset["query"],
    dataset["context"],
    dataset["answer"],
    chunk_sizes,
    chunk_num,
    models,
    judges,
    embedding_function,
    metrics=[context_precision],
    splitter=RecursiveCharacterTextSplitter,
    repeat_ragas_eval=REPEAT
)

In [ ]:
result

In [5]:
import pickle
METRICS_DIR = "metrics_results"
with open(f"{METRICS_DIR}/context_precision_hf.pickle", "wb") as handle:
    pickle.dump(result, handle)